# Public-data audit starter

Unexecuted setup notebook. This checks locally cached MLB feed structure only. It does not reconstruct runner opportunities or validate release timestamps. Save public game feed JSON files under `data/raw/` and document their source URLs, retrieval dates, and checksums.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
RAW = ROOT / 'data' / 'raw'
feed_files = sorted(RAW.glob('*.json'))
print(f'{len(feed_files)} cached JSON files found')

In [ ]:
rows = []
for path in feed_files:
    feed = json.loads(path.read_text(encoding='utf-8'))
    if 'liveData' not in feed:
        raise ValueError(f'{path.name}: not a live game feed')
    for play in feed['liveData'].get('plays', {}).get('allPlays', []):
        for event in play.get('playEvents', []):
            rows.append({
                'source_file': path.name,
                'game_pk': feed.get('gamePk'),
                'at_bat_index': play.get('about', {}).get('atBatIndex'),
                'event_index': event.get('index'),
                'is_pitch': event.get('isPitch', False),
                'event_type': event.get('details', {}).get('eventType'),
                'start_time': event.get('startTime'),
                'end_time': event.get('endTime'),
            })
events = pd.DataFrame(rows)
print(f'{len(events)} events extracted')

In [ ]:
if not events.empty:
    pitches = events.loc[events['is_pitch'].eq(True)].copy()
    for column in ['start_time', 'end_time']:
        pitches[column] = pd.to_datetime(pitches[column], utc=True, errors='coerce')
    display(pitches[['start_time', 'end_time']].isna().sum().rename('missing_or_invalid'))
    keys = ['game_pk', 'at_bat_index', 'event_index']
    print('Duplicate pitch keys:', pitches.duplicated(keys).sum())
    display(events['event_type'].value_counts(dropna=False))
else:
    print('No feeds loaded. No audit results have been produced.')

## Before calculating intervals

Validate event order and reconstruct pre-action state. Inventory intervening events and manually validate public-video release intervals. Timestamp formatting alone establishes neither accuracy nor release alignment. Follow `docs/study_plan.md`; retain nonpitch records and opportunities ending without another pitch.